In [5]:
import json
import uuid
import time
import calendar
from datetime import datetime
import pandas as pd
import numpy as np
import _utils_

last_updated_date_epoch = int(time.time()*1000)
print("last_updated_date_epoch",last_updated_date_epoch) 

# Empieza Febrero 2007
published_date = datetime(2007, 2, 1, 1, 0, 0)
published_date_epoch = calendar.timegm(published_date.timetuple())*1000
print("published_date_epoch", published_date_epoch) 

last_updated_date_epoch 1782480271572
published_date_epoch 1170291600000


In [8]:
zone_fname = "inputs/zone.json"
zone_data = json.load(open(zone_fname))
policy_fname = "inputs/policy.csv"
policy_table  = pd.read_csv(policy_fname)
policy_table = policy_table.astype(object)

# Replace all NaNs with None
policy_table = policy_table.where(pd.notnull(policy_table), None)

policy_table["curb_policy_id"] = str(uuid.uuid4())
policy_table['time_spans.days_of_week'] = policy_table['time_spans.days_of_week'].apply(lambda x: x.split("-"))
policy_table['published_date'] = published_date_epoch
policy_table

,curb_policy_int_id,published_date,priority,data_source_operator_id,time_spans.days_of_week,time_spans.time_of_day_start,time_spans.time_of_day_end,rules.activity,rules.max_stay,rules.user_classes,rates.duration,rates.rate,curb_policy_id
0,2,1170291600000,3000,None,"[mon, tue, wed, thu, fri]",0:00,23:59,no paking,None,all,0,0.0,f6b7cac0-10e2-4e0d-83e7-c58173c9fd9c
1,1,1170291600000,5000,None,"[mon, tue, wed, thu, fri, sat, sun]",8:00,18:00,parking,180.0,all,30,0.5,f6b7cac0-10e2-4e0d-83e7-c58173c9fd9c
2,3,1170291600000,1000,None,"[mon, tue, wed, thu, fri, sat, sun]",8:00,18:00,accessible parking,360.0,accessible,60,0.4,f6b7cac0-10e2-4e0d-83e7-c58173c9fd9c


In [ ]:

# Apply to all rows
cds_policies = [_utils_.row_to_nested_dict(row) for _, row in policy_table.iterrows()]

# print(json.dumps(nested_policies, indent=2))
for policy in cds_policies:
    policy['rules'] = [policy['rules'] ]
    policy['rates'] = [policy['rates'] ]
# print(json.dumps(nested_policies, indent=2))

policy_final_data = _utils_.get_metadata()
policy_final_data["last_updated"] = last_updated_date_epoch
policy_final_data["data"] = {
    "policies" : cds_policies
}
with open("out_cds/cds_policy_ambato.json", "w") as f:
    json.dump(policy_final_data, f, indent=2)

### Zones

In [10]:
cds_zones = []
for zone in zone_data["features"]:
    policy_number = zone["properties"]["policy"]
    policy_uuid  = list(policy_table[policy_table["curb_policy_int_id"]==int(policy_number)]["curb_policy_id"].values)

    cds_zone = {
        "curb_zone_id": str(uuid.uuid4()),
        "geometry": zone["geometry"],
        "curb_policy_ids":policy_uuid,
        "published_date": published_date_epoch,
        "last_updated_date": last_updated_date_epoch,
        "start_date": published_date_epoch
    }
    cds_zones.append(cds_zone)


zone_final_data = _utils_.get_metadata()
zone_final_data["last_updated"] = last_updated_date_epoch
zone_final_data["data"] = {
    "zones" : cds_zones
}

In [ ]:
with open("out_cds/cds_zone_ambato.json", "w") as f:
    json.dump(zone_final_data, f, indent=2)